In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

torch.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [3]:
if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()

chars = sorted(list(set(text)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("text length:", len(text))
print("vocab_size:", vocab_size)
print(text[:500])

text length: 1115394
vocab_size: 65
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [4]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


block_size = 64

dataset = NextTokenDataset(data, block_size)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

xb, yb = next(iter(loader))

print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

xb.shape: torch.Size([64, 64])
yb.shape: torch.Size([64, 64])


In [8]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()

        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(wei, dim=-1)

        out = wei @ v

        return out


class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)

        self.attn = SingleHeadSelfAttention(emb_dim, block_size)

        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape

        pos = torch.arange(T, device=x.device)

        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]

        h = tok + pos
        h = self.attn(h)

        logits = self.lm_head(h)

        return logits


model = TinyAttentionLM(vocab_size, block_size)

logits = model(xb)

print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 64, 65])


In [7]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()

        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        out = wei @ v

        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()

        head_size = emb_dim // num_heads

        self.heads = nn.ModuleList([
            Head(emb_dim, head_size, block_size, dropout)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(emb_dim, emb_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.proj(out)

        out = self.dropout(out)

        return out


class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(emb_dim)

        self.sa = MultiHeadAttention(
            emb_dim,
            num_heads,
            block_size,
            dropout
        )

        self.ln2 = nn.LayerNorm(emb_dim)

        self.ffwd = FeedForward(
            emb_dim,
            dropout
        )

    def forward(self, x):
        x = x + self.sa(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x


class TinyGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        emb_dim=128,
        num_heads=4,
        num_layers=4,
        dropout=0.1
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            emb_dim
        )

        self.position_embedding = nn.Embedding(
            block_size,
            emb_dim
        )

        self.blocks = nn.Sequential(*[
            Block(
                emb_dim,
                num_heads,
                block_size,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(emb_dim)

        self.lm_head = nn.Linear(
            emb_dim,
            vocab_size
        )

    def forward(self, x):
        B, T = x.shape

        pos = torch.arange(
            T,
            device=x.device
        )

        tok = self.token_embedding(x)

        pos = self.position_embedding(pos)[None]

        h = tok + pos

        h = self.blocks(h)

        h = self.ln_f(h)

        logits = self.lm_head(h)

        return logits


model = TinyGPT(vocab_size, block_size)

logits = model(xb)

print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 64, 65])


In [11]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(
        logits.transpose(1, 2),
        targets
    )


def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()

    total_loss = 0.0

    total_count = 0

    for step, (xb, yb) in enumerate(loader):
        xb = xb.to(device)

        yb = yb.to(device)

        logits = model(xb)

        loss = sequence_cross_entropy(
            logits,
            yb
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * xb.size(0)

        total_count += xb.size(0)

        if max_steps is not None and step + 1 >= max_steps:
            break

    return total_loss / total_count


model = TinyGPT(vocab_size, block_size).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

print("model created")
print(device)

model created
cpu


In [12]:
for epoch in range(30):
    train_loss = train_one_epoch(
        model,
        loader,
        optimizer,
        device,
        max_steps=300
    )

    print(
        f"epoch {epoch:2d} | train loss {train_loss:.4f}"
    )

epoch  0 | train loss 2.6826
epoch  1 | train loss 2.3131
epoch  2 | train loss 2.1449
epoch  3 | train loss 2.0270
epoch  4 | train loss 1.9387
epoch  5 | train loss 1.8661
epoch  6 | train loss 1.8063
epoch  7 | train loss 1.7638
epoch  8 | train loss 1.7286
epoch  9 | train loss 1.6960
epoch 10 | train loss 1.6702
epoch 11 | train loss 1.6468
epoch 12 | train loss 1.6287
epoch 13 | train loss 1.6104
epoch 14 | train loss 1.5949
epoch 15 | train loss 1.5825
epoch 16 | train loss 1.5712
epoch 17 | train loss 1.5587
epoch 18 | train loss 1.5494
epoch 19 | train loss 1.5393
epoch 20 | train loss 1.5274
epoch 21 | train loss 1.5221
epoch 22 | train loss 1.5147
epoch 23 | train loss 1.5061
epoch 24 | train loss 1.5033
epoch 25 | train loss 1.4966
epoch 26 | train loss 1.4885
epoch 27 | train loss 1.4812
epoch 28 | train loss 1.4735
epoch 29 | train loss 1.4752


In [13]:
@torch.no_grad()
def sample_gpt(
    model,
    block_size,
    stoi,
    itos,
    device,
    start_text="",
    max_new_tokens=400
):
    model.eval()

    context = torch.zeros(
        (1, block_size),
        dtype=torch.long,
        device=device
    )

    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor(
                [[stoi[ch]]],
                device=device
            )

            context = torch.cat(
                [context[:, 1:], ix],
                dim=1
            )

    out = list(start_text)

    for _ in range(max_new_tokens):
        logits = model(context)

        logits = logits[:, -1, :]

        probs = F.softmax(
            logits,
            dim=-1
        )

        ix = torch.multinomial(
            probs,
            num_samples=1
        )

        out.append(
            itos[ix.item()]
        )

        context = torch.cat(
            [context[:, 1:], ix],
            dim=1
        )

    return "".join(out)


print(
    sample_gpt(
        model,
        block_size,
        stoi,
        itos,
        device,
        start_text="",
        max_new_tokens=500
    )
)

DUCHESS OF, be conducted:
Farewell?

Second Citizen:
Come, let's than I love him, a nickle.

JULIET:
But no the noble viance of mine to so preserve of mine;
And, lork thus kind, that you for you sit might friend,
And but beseech equain bleg's partiant sun,
Lorders, to revenge him Paulina hidst hopes,
But a mawker comes is, if and thineven live out.

RICHMOND:
And with devising it, but their twench'd the housest?

AUFIDIUS:
Here? An I never dig tretcher's,
The trays Duke-long you did father.

KIN


In [14]:
text = """
일찍이 백성의 재물은 털끝만큼도 뺏은 적이 없고, 수령의 뇌물과 불의한 놈의 재물을 빼앗아 먹고, 간혹 나라 곡식을 도적하기는 했으나 임금과 아비가 한몸이니 자식이 아비 것을 좀 먹었다고 도적이라 하겠사옵니까? 어린 자식이 어미 젖 먹는 것과 마찬가지이옵니다.
"""

In [20]:
block_size = 32

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [21]:
model = TinyGPT(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(30):
    train_loss = train_one_epoch(
        model,
        loader,
        optimizer,
        device,
        max_steps=300
    )
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

print(sample_gpt(
    model,
    block_size,
    stoi,
    itos,
    device,
    start_text="일찍이 백성의 재물은",
    max_new_tokens=200
))

epoch  0 | train loss 4.0425
epoch  1 | train loss 3.5169
epoch  2 | train loss 3.3454
epoch  3 | train loss 3.1657
epoch  4 | train loss 3.0168
epoch  5 | train loss 2.8409
epoch  6 | train loss 2.6644
epoch  7 | train loss 2.4892
epoch  8 | train loss 2.3198
epoch  9 | train loss 2.1566
epoch 10 | train loss 1.9939
epoch 11 | train loss 1.8485
epoch 12 | train loss 1.7201
epoch 13 | train loss 1.6030
epoch 14 | train loss 1.5029
epoch 15 | train loss 1.4160
epoch 16 | train loss 1.3409
epoch 17 | train loss 1.2735
epoch 18 | train loss 1.2090
epoch 19 | train loss 1.1491
epoch 20 | train loss 1.0898
epoch 21 | train loss 1.0297
epoch 22 | train loss 0.9684
epoch 23 | train loss 0.9105
epoch 24 | train loss 0.8523
epoch 25 | train loss 0.7992
epoch 26 | train loss 0.7439
epoch 27 | train loss 0.6923
epoch 28 | train loss 0.6397
epoch 29 | train loss 0.5937
일찍이 백성의 재물은 없고, 어것을 없고, 빼앗아비가 도적하기는 젖 빼앗아 재물을 임금과 했으나 고, 한찬니까? 자식을 아비 먹는 먹었미 도적하기는 했으나라 자식이니 했으나 아비의 큼하겠사옵니 먹었다고 자식이 고, 아비 것을 도적이니